In [1]:
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np 

# On importe un graphe pour chacune des université
G_caltech = nx.read_gml('./db/Caltech36.gml')
G_mit = nx.read_gml('./db/MIT8.gml')
G_hopkins = nx.read_gml('./db/Johns Hopkins55.gml')

# Question 5
On cherche maintenant à trouver des étiquettes manquantes sur un graphe, et à calculer la précision de cette recherche.

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch_geometric.nn import GATConv, GCNConv
from torch_geometric.utils import from_networkx, train_test_split_edges, to_undirected
from torch_geometric.data import Data
from sklearn.preprocessing import OneHotEncoder

In [3]:
# Les paramètres de notre algorithme
max_iter = 100
eps = 1e-3
alpha = 0.5
data = "data/MIT8.gml"
iterations = 10

def score_label_propagation_algorithm(attribute, remove_fraction):
    score = []
    for _ in range(iterations):
        graph = nx.read_gml('./db/Johns Hopkins55.gml')

        # On supprime un certain pourcentage de nos liens
        nodes = list(graph.nodes())
        np.random.shuffle(nodes)
        removed = []
        for node in nodes[:int(len(nodes) * remove_fraction)]:
            if attribute in graph.nodes[node]:
                removed.append(node)
                graph.nodes[node][attribute] = None  # Utilisation de None pour marquer les labels manquants

        # On calcule les matrice d'adjacence + normalisation
        a = nx.adjacency_matrix(graph)
        d_inv_sqrt = np.diag([1/np.sqrt(x) if x != 0 else 0 for x in np.sum(a, axis=1)])
        s = d_inv_sqrt @ a @ d_inv_sqrt

        # OneHot nous donne les étiquettes
        encoder = OneHotEncoder()
        y_true = [[G_hopkins.nodes[node][attribute]] for node in graph]
        y_true = encoder.fit_transform(y_true)
        y_init = [[graph.nodes[node][attribute] if graph.nodes[node][attribute] is not None else 0] for node in graph]
        y_init = encoder.transform(y_init)

        y = y_init.copy()

        # On propage les étiquettes
        for _ in range(max_iter):
            y_new = alpha * s @ y + (1 - alpha) * y_init
            if np.linalg.norm(y - y_new, 1) < eps:
                break
            y = y_new

        # On calcule le score pour pouvoir mesurer la précision de notre alogrithme
        success, failures = 0, 0
        for idx, node in enumerate(graph):
            if node in removed:
                if np.argmax(y[idx]) == np.argmax(y_true[idx]):
                    success += 1
                else:
                    failures += 1
        score.append(success / (success + failures))
    return score

In [4]:
# On affiche le score pour chaque cirtère et pourcentage de suppressions d'étiquettes
for attribute in ["major_index", "dorm", "gender"]:
    for remove_fraction in [0.1, 0.2, 0.3]:
        score = score_label_propagation_algorithm(attribute, remove_fraction)
        print(f"Attribute: {attribute}, remove fraction: {remove_fraction}, score: {np.mean(score)}")

Attribute: major_index, remove fraction: 0.1, score: 0.13532818532818533
Attribute: major_index, remove fraction: 0.2, score: 0.13359073359073362
Attribute: major_index, remove fraction: 0.3, score: 0.13384813384813385
Attribute: dorm, remove fraction: 0.1, score: 0.4115830115830115
Attribute: dorm, remove fraction: 0.2, score: 0.4100386100386101
Attribute: dorm, remove fraction: 0.3, score: 0.41113256113256114
Attribute: gender, remove fraction: 0.1, score: 0.08204633204633205
Attribute: gender, remove fraction: 0.2, score: 0.08175675675675675
Attribute: gender, remove fraction: 0.3, score: 0.07882882882882883


# Interprétation
Les scores que j’ai obtenus avec l’algorithme de label propagation sont globalement plus faibles que ceux donnés dans le sujet, surtout pour le genre.

Je pense que ça peut venir du nombre d’itérations que j’ai fixé à 100. Peut-être qu’il n’est pas suffisant pour bien faire converger la propagation des labels, surtout sur certaines classes peu connectées.
Le choix du graphe utilisé peut aussi expliquer les écarts. Le sujet utilise les données de Duke, alors que moi j’ai pris MIT, et on a vu que certaines population ont des réseaux différents selon des critères différents.

Certains labels sont aussi plus faciles à prédire car ils sont plus fortement corrélés à la structure du graphe. Par exemple, les étudiants vivant dans le même dortoir sont souvent connectés entre eux, ce qui facilite la propagation correcte de l’attribut dorm, contrairement à gender qui peut être plus réparti aléatoirement dans le réseau.